In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 44. Week 30 — Hierarchical shrinkage and model-evaluation boundaries

## 学習目標

- no pooling、complete pooling、partial poolingを比較できる
- sampling standard errorに応じたshrinkageを説明できる
- tenor exchangeabilityの仮定を監査できる
- WAICのpointwise-i.i.d.近似を時系列へ無批判に適用しない

## 前提知識

- Week 29のconjugacyとposterior predictive
- B3のestimandとdependence-aware inference

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 44


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Normal-normal partial pooling

$$
\hat\theta_j\mid\theta_j\sim N(\theta_j,s_j^2),\qquad
\theta_j\mid\mu,\tau^2\sim N(\mu,\tau^2).
$$

posterior meanは (w_j\hat\theta_j+(1-w_j)\mu)、(w_j=\tau^2/(\tau^2+s_j^2))。本章は (mu,\tau) をpredeclared sensitivity parameterとして固定し、empirical Bayes推定と取り違えない。

In [4]:
horizon = 5
origins = np.arange(curve_yields.shape[0] - horizon)
targets = (curve_yields[origins + horizon] - curve_yields[origins]) * 100.0
target_dates = curve_dates[origins + horizon]
training_rows = target_dates <= train_end_date
training_targets = targets[training_rows]
raw_means = training_targets.mean(axis=0)
standard_errors = training_targets.std(axis=0, ddof=1) / np.sqrt(training_targets.shape[0])
pooled = qt.hierarchical_normal_posterior(
    raw_means,
    standard_errors,
    population_mean=0.0,
    population_standard_deviation=1.0,
)
pooling_table = pd.DataFrame(
    {
        "tenor": qt.DEFAULT_TENORS,
        "no_pooling_mean_bp": raw_means,
        "standard_error_bp": standard_errors,
        "partial_pooling_mean_bp": pooled.means,
        "shrinkage_weight": pooled.shrinkage_weights,
        "complete_pooling_mean_bp": np.average(raw_means, weights=1.0 / standard_errors**2),
    }
)
display(pooling_table)

fig = go.Figure()
fig.add_scatter(x=pooling_table["tenor"], y=pooling_table["no_pooling_mean_bp"], name="no pooling", mode="markers")
fig.add_scatter(x=pooling_table["tenor"], y=pooling_table["partial_pooling_mean_bp"], name="partial pooling", mode="markers")
fig.update_layout(title="Five-publication mean changes before and after partial pooling", yaxis_title="Mean change (bp)", template="plotly_white")
fig.show()

,tenor,no_pooling_mean_bp,standard_error_bp,partial_pooling_mean_bp,shrinkage_weight,complete_pooling_mean_bp
0,3m,0.007879,0.154269,0.007696,0.976754,-0.119547
1,2y,-0.127879,0.177620,-0.123968,0.969416,-0.119547
2,5y,-0.226667,0.225539,-0.215695,0.951595,-0.119547
3,10y,-0.214545,0.237140,-0.203123,0.946759,-0.119547
4,30y,-0.192727,0.237716,-0.182419,0.946513,-0.119547


## 2. Tenor-specific predictive coverage

In [5]:
raw_features = np.column_stack(
    [curve_yields[origins], np.vstack([np.zeros(5), np.diff(curve_yields, axis=0)])[origins] * 100.0]
)
feature_mean = raw_features[training_rows].mean(axis=0)
feature_scale = raw_features[training_rows].std(axis=0, ddof=1)
design = np.column_stack([np.ones(raw_features.shape[0]), (raw_features - feature_mean) / feature_scale])
validation_rows = (target_dates > train_end_date) & (target_dates <= validation_end_date)
predictive_rows = []
models = []
for tenor_index, tenor in enumerate(qt.DEFAULT_TENORS):
    model = qt.fit_bayesian_linear_regression(
        design[training_rows], targets[training_rows, tenor_index], prior_precision=1.0, prior_shape=2.0, prior_scale=25.0
    )
    models.append(model)
    predictive = qt.bayesian_linear_predictive(model, design[validation_rows])
    lower, upper = predictive.interval(0.9)
    actual = targets[validation_rows, tenor_index]
    predictive_rows.append(
        {
            "tenor": tenor,
            "rmse_bp": np.sqrt(np.mean((actual - predictive.mean) ** 2)),
            "coverage_90": np.mean((actual >= lower) & (actual <= upper)),
            "mean_width_bp": np.mean(upper - lower),
        }
    )
display(pd.DataFrame(predictive_rows))

,tenor,rmse_bp,coverage_90,mean_width_bp
0,3m,11.087630,0.749543,19.042536
1,2y,18.183303,0.526508,23.472860
2,5y,17.981122,0.614260,30.141091
3,10y,16.630609,0.650823,31.574337
4,30y,15.013069,0.685558,31.463550


## 3. WAIC diagnostic and dependence boundary

WAICはdraw-by-observation log likelihoodから

$$
\operatorname{WAIC}=-2\left(\sum_i\log E_s[p(y_i\mid\theta_s)]-\sum_i\operatorname{Var}_s[\log p(y_i\mid\theta_s)]\right)
$$

を計算する。ただしoverlapping 5公表日targetは独立でないため、naive pointwise WAICを最終model selectorにしない。

In [6]:
waic_rng = task_rng(2)
model = models[3]
precision_inverse = np.linalg.solve(model.precision, np.eye(model.precision.shape[0]))
draw_count = 500
sigma_squared = model.scale / waic_rng.gamma(model.shape, 1.0, size=draw_count)
beta_draws = np.vstack(
    [waic_rng.multivariate_normal(model.mean, sigma_squared[index] * precision_inverse) for index in range(draw_count)]
)
audit_design = design[training_rows][-400:]
audit_target = targets[training_rows, 3][-400:]
log_likelihood_draws = np.empty((draw_count, audit_target.size))
for draw in range(draw_count):
    residual = audit_target - audit_design @ beta_draws[draw]
    log_likelihood_draws[draw] = -0.5 * (
        np.log(2.0 * np.pi * sigma_squared[draw]) + residual**2 / sigma_squared[draw]
    )
waic_value, effective_parameters = qt.waic(log_likelihood_draws)
print("naive pointwise WAIC diagnostic:", waic_value)
print("effective parameter diagnostic:", effective_parameters)
print("used for time-series model selection:", False)

naive pointwise WAIC diagnostic: 2985.151967326189
effective parameter diagnostic: 7.631173077999387
used for time-series model selection: False


## 4. 失敗モード

- tenorがexchangeableかを確認せずpoolingする
- (mu,	au) をdataから選んでfixed priorと呼ぶ
- shrinkageをbias-freeと表現する
- overlapping targetsへnaive i.i.d. WAICをmodel selectorとして使う
- interval widthを隠してcoverageだけを比較する

## 5. 段階別演習

### 基礎

1. shrinkage weightの極限 (s_j\to0,\infty) を説明せよ。
2. no/complete/partial poolingを図示せよ。

### 標準

3. (	au=0.25,1,5) bpの感応度を比較せよ。
4. tenor階層ではなくmaturity spline priorを設計せよ。

### 研究

5. leave-future-block-out predictive evaluationを設計せよ。

## 6. Exit Criteria

- [ ] three pooling regimesを区別した
- [ ] hyperparameterの固定/推定を明記した
- [ ] tenor exchangeabilityを仮定として書いた
- [ ] RMSE、coverage、widthをtenor別に報告した
- [ ] WAICのtime-series dependence boundaryを明記した

## 7. 出典


- [Gelman et al., Bayesian Data Analysis, 3rd ed.](https://sites.stat.columbia.edu/gelman/book/)
- [Gelman et al., Bayesian Workflow](https://arxiv.org/abs/2011.01808)
- [Vehtari, Gelman, and Gabry, Practical Bayesian model evaluation](https://doi.org/10.1007/s11222-016-9696-4)